# 90 — Scoring

Loads every processed variable, min-max normalizes each to `[0, 1]`, and combines them using `weights.yaml` into a single livability score.

In [1]:
from common import load_layers, load_weights, normalize, save_variable, weighted_score

## Weights

Raw values from `weights.yaml`, zeros dropped. Renormalization happens after we know which layers actually loaded.

In [2]:
weights = load_weights()
weights

{'sea_proximity': 1.0,
 'terrain_ruggedness': 1.0,
 'sun_hours': 1.0,
 'temperature_pleasantness': 1.0,
 'annual_greenness': 1.0,
 'precipitation_balance': 1.0,
 'climate_vulnerability': 1.0,
 'natural_disaster_risk': 1.0,
 'air_quality': 1.0,
 'population_density': 0.5}

## Load layers

Each layer is `processed/<name>.nc`. Missing layers are skipped with a warning so scoring works incrementally as more variables are implemented.

In [3]:
layers = load_layers(weights)
print(f'loaded {len(layers)} of {len(weights)} layers')

loaded 10 of 10 layers


## Normalize

In [4]:
normed = {name: normalize(da) for name, da in layers.items()}

## Weighted sum

Per-cell weighted average. If a layer is missing at a cell, its weight is redistributed across the layers that do have data there — so a missing variable neither drags the score down nor discards the cell. A cell is only NaN where every layer is missing.

In [5]:
score = weighted_score(normed, weights)
score.name = 'livability'

## Save

In [6]:
out = save_variable(score, 'score')
print(f'wrote {out}')

wrote /Users/lutz/Documents/alfatraining/projects/data-science-notebooks/data/world-livable-atlas/processed/score.nc
